# RQ5: Chunking Strategies and Reranking for RAG

**Deep Learning · Master Wirtschaftsinformatik · Hochschule Heilbronn**

**Research Question:** *How do chunking strategy (size, overlap, method) and cross-encoder reranking influence retrieval quality and answer accuracy of a RAG pipeline?*

---

In the lectures on Context Engineering and RAG, we learned that retrieval quality depends critically on how documents are split into chunks. This notebook provides the scaffolding for a systematic investigation.

**What you will do:**

1. Load a document corpus and create an evaluation dataset
2. Implement three chunking strategies (fixed-size, sentence-based, recursive)
3. Embed chunks and build a FAISS index for each configuration
4. Retrieve and evaluate using Recall@k and MRR
5. Add cross-encoder reranking and measure improvement
6. Analyze failure patterns

**Your tasks are marked with `# TODO` — search for them to find all places where you need to add code.**

---
## Setup

In [ ]:
# Install required packages (uncomment if needed)
# !pip install sentence-transformers faiss-cpu transformers datasets nltk

import numpy as np
import pandas as pd
import faiss
import time
import re
from collections import defaultdict
from sentence_transformers import SentenceTransformer, CrossEncoder
from datasets import load_dataset
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)

print("Setup complete.")

---
## Part 1: Load Corpus and Create Evaluation Dataset

Choose **one** of the two corpus options below. Uncomment the option you want to use and comment out the other.

**Option A: Wikipedia AI/DL Corpus** (recommended) — 20 full Wikipedia articles on AI/DL topics with 40 provided evaluation questions. Self-referencing: you build a RAG system about the topics you study.

**Option B: QASPER** — NLP research papers with evidence-annotated QA pairs. Subset to 30–50 papers for manageable runtimes.

In [ ]:
# ============================================================
# OPTION A: Wikipedia AI/DL Corpus (recommended)
# ============================================================
# Requires the corpus/ folder with wikipedia_dl_corpus.json and wikipedia_dl_questions.json

import json, os

corpus_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'corpus')

with open(os.path.join(corpus_dir, 'wikipedia_dl_corpus.json')) as f:
    corpus_data = json.load(f)
with open(os.path.join(corpus_dir, 'wikipedia_dl_questions.json')) as f:
    questions_data = json.load(f)

# Documents: list of full article texts
documents = [doc['text'] for doc in corpus_data]
doc_titles = [doc['title'] for doc in corpus_data]
doc_ids = [doc['id'] for doc in corpus_data]

# Evaluation data: questions with ground truth
eval_data = []
for q in questions_data:
    # Find the source document index
    src_idx = next(i for i, d in enumerate(corpus_data) if d['id'] == q['source_article'])
    eval_data.append({
        'question': q['question'],
        'gold_context': documents[src_idx],
        'gold_doc_idx': src_idx,
        'expected_answer': q['answer'],
        'difficulty': q['difficulty'],
    })

print(f"Corpus: {len(documents)} documents, {sum(len(d.split()) for d in documents):,} words")
print(f"Evaluation: {len(eval_data)} questions ({sum(1 for e in eval_data if e['difficulty']=='easy')} easy, "
      f"{sum(1 for e in eval_data if e['difficulty']=='medium')} medium, "
      f"{sum(1 for e in eval_data if e['difficulty']=='hard')} hard)")
print(f"\nSample: Q: {eval_data[0]['question']}")
print(f"        A: {eval_data[0]['expected_answer'][:100]}")

In [ ]:
# ============================================================
# OPTION B: QASPER (uncomment this block, comment out Option A)
# ============================================================
# Requires: pip install datasets

# from datasets import load_dataset
# qasper = load_dataset("allenai/qasper", split="test")
#
# # Subset to 30-50 papers for manageable experiment runtimes
# N_PAPERS = 40
# qasper_subset = qasper.select(range(min(N_PAPERS, len(qasper))))
#
# # Extract full paper text by concatenating all section paragraphs
# documents = []
# doc_titles = []
# eval_data = []
#
# for paper_idx, paper in enumerate(qasper_subset):
#     # Concatenate all section texts into one document
#     full_text_parts = []
#     for section_name, paragraphs in zip(paper['full_text']['section_name'],
#                                          paper['full_text']['paragraphs']):
#         if section_name:
#             full_text_parts.append(f"\n## {section_name}\n")
#         full_text_parts.extend(paragraphs)
#     full_text = '\n'.join(full_text_parts)
#
#     if len(full_text.split()) < 500:  # skip very short papers
#         continue
#
#     documents.append(full_text)
#     doc_titles.append(paper['title'])
#
#     # Extract QA pairs with evidence
#     for qa in paper['qas']:
#         question = qa['question']
#         for answer_info in qa['answers']:
#             answer_text = answer_info.get('answer', {}).get('free_form_answer', '')
#             evidence = answer_info.get('answer', {}).get('evidence', [])
#             if answer_text and answer_text.strip():
#                 eval_data.append({
#                     'question': question,
#                     'gold_context': full_text,
#                     'gold_doc_idx': len(documents) - 1,
#                     'expected_answer': answer_text,
#                     'difficulty': 'medium',
#                 })
#                 break  # one answer per question is enough
#
# print(f"Corpus: {len(documents)} papers, {sum(len(d.split()) for d in documents):,} words")
# print(f"Evaluation: {len(eval_data)} questions")
# print(f"\nSample: Q: {eval_data[0]['question']}")
# print(f"        A: {eval_data[0]['expected_answer'][:100]}")

---
## Part 2: Chunking Strategies

We implement three chunking methods. Each takes a document and returns a list of chunks.

**Your job:** understand these implementations, then run them across different configurations.

In [ ]:
def chunk_fixed_size(text, chunk_size=256, overlap=0):
    """Split text into fixed-size chunks (by characters).

    Args:
        text: Input document text
        chunk_size: Number of characters per chunk
        overlap: Number of overlapping characters between chunks

    Returns:
        List of chunk strings
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return [c.strip() for c in chunks if c.strip()]


def chunk_sentence_based(text, chunk_size=256, overlap=0):
    """Split text into chunks respecting sentence boundaries.

    Groups sentences until reaching approximately chunk_size characters.
    """
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_len = 0

    for sent in sentences:
        sent_len = len(sent)
        if current_len + sent_len > chunk_size and current_chunk:
            chunks.append(' '.join(current_chunk))
            # Handle overlap: keep last sentence(s) for context
            if overlap > 0:
                overlap_sents = []
                overlap_len = 0
                for s in reversed(current_chunk):
                    if overlap_len + len(s) <= overlap:
                        overlap_sents.insert(0, s)
                        overlap_len += len(s)
                    else:
                        break
                current_chunk = overlap_sents
                current_len = overlap_len
            else:
                current_chunk = []
                current_len = 0
        current_chunk.append(sent)
        current_len += sent_len

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return [c.strip() for c in chunks if c.strip()]


def chunk_recursive(text, chunk_size=256, overlap=0):
    """Recursive character splitting: try paragraph -> sentence -> word boundaries.

    Attempts to split at the most natural boundary that keeps chunks under chunk_size.
    This is the approach used by LangChain's RecursiveCharacterTextSplitter.
    """
    separators = ['\n\n', '\n', '. ', ' ']

    def _split(text, separators):
        if len(text) <= chunk_size:
            return [text]

        # Find the best separator
        sep = separators[0] if separators else ' '
        for s in separators:
            if s in text:
                sep = s
                break

        parts = text.split(sep)
        chunks = []
        current = []
        current_len = 0

        for part in parts:
            part_with_sep = part + sep
            if current_len + len(part_with_sep) > chunk_size and current:
                chunk_text = sep.join(current)
                if len(chunk_text) > chunk_size and len(separators) > 1:
                    # Recursively split with finer separator
                    chunks.extend(_split(chunk_text, separators[1:]))
                else:
                    chunks.append(chunk_text)

                if overlap > 0:
                    # Keep some context
                    overlap_parts = []
                    ol = 0
                    for p in reversed(current):
                        if ol + len(p) <= overlap:
                            overlap_parts.insert(0, p)
                            ol += len(p)
                        else:
                            break
                    current = overlap_parts
                    current_len = ol
                else:
                    current = []
                    current_len = 0

            current.append(part)
            current_len += len(part_with_sep)

        if current:
            remaining = sep.join(current)
            if len(remaining) > chunk_size and len(separators) > 1:
                chunks.extend(_split(remaining, separators[1:]))
            else:
                chunks.append(remaining)

        return [c.strip() for c in chunks if c.strip()]

    return _split(text, separators)


# Test chunking methods on a sample document
sample_doc = documents[0]
print(f"Sample document ({len(sample_doc)} chars):\n{sample_doc[:300]}...\n")

for method_name, method in [("Fixed-size", chunk_fixed_size),
                             ("Sentence-based", chunk_sentence_based),
                             ("Recursive", chunk_recursive)]:
    chunks = method(sample_doc, chunk_size=256, overlap=0)
    print(f"{method_name}: {len(chunks)} chunks, avg {np.mean([len(c) for c in chunks]):.0f} chars")

---
## Part 3: Embedding, Indexing, and Retrieval

We use a Sentence Transformer model to embed chunks and FAISS for fast retrieval.

In [ ]:
# Load embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Embedding model: all-MiniLM-L6-v2")
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

In [ ]:
def build_index_and_retrieve(documents, queries, chunk_fn, chunk_size, overlap,
                              embed_model, k=10):
    """Build a FAISS index from chunked documents and retrieve for each query.

    Returns:
        chunks: list of all chunks
        chunk_to_doc: mapping from chunk index to source document index
        results: list of (query_idx, retrieved_chunk_indices, scores) tuples
        index_time: time to chunk + embed + index
        query_time: time to retrieve all queries
    """
    # Chunk all documents
    t0 = time.time()
    all_chunks = []
    chunk_to_doc = []
    for doc_idx, doc in enumerate(documents):
        doc_chunks = chunk_fn(doc, chunk_size=chunk_size, overlap=overlap)
        for chunk in doc_chunks:
            all_chunks.append(chunk)
            chunk_to_doc.append(doc_idx)

    # Embed chunks
    chunk_embeddings = embed_model.encode(all_chunks, show_progress_bar=False,
                                          normalize_embeddings=True)

    # Build FAISS index
    dim = chunk_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product = cosine for normalized vectors
    index.add(chunk_embeddings.astype('float32'))
    index_time = time.time() - t0

    # Retrieve for each query
    t0 = time.time()
    query_embeddings = embed_model.encode(queries, show_progress_bar=False,
                                           normalize_embeddings=True)
    scores, indices = index.search(query_embeddings.astype('float32'), k)
    query_time = time.time() - t0

    results = []
    for q_idx in range(len(queries)):
        results.append({
            'query_idx': q_idx,
            'retrieved_indices': indices[q_idx].tolist(),
            'scores': scores[q_idx].tolist(),
        })

    return all_chunks, chunk_to_doc, results, index_time, query_time


# Quick test
queries = [ex['question'] for ex in eval_data]
chunks, c2d, results, t_idx, t_q = build_index_and_retrieve(
    documents, queries, chunk_fixed_size, chunk_size=256, overlap=0,
    embed_model=embed_model, k=5
)
print(f"Chunks: {len(chunks)}, Index time: {t_idx:.2f}s, Query time: {t_q:.2f}s")
print(f"\nSample retrieval for: '{queries[0]}'")
print(f"  Top chunk: {chunks[results[0]['retrieved_indices'][0]][:150]}...")

---
## Part 4: Evaluation Metrics

We implement Recall@k and MRR to measure retrieval quality.

In [ ]:
def compute_metrics(eval_data, all_chunks, chunk_to_doc, results, k_values=[5, 10]):
    """Compute Recall@k and MRR for retrieval results.

    A retrieved chunk is 'relevant' if it comes from the same source document
    as the gold context (i.e., the chunk contains part of the answer context).
    Uses 'gold_doc_idx' from eval_data to identify the source document.
    """
    metrics = {}

    for k in k_values:
        recalls = []
        reciprocal_ranks = []

        for q_idx, res in enumerate(results):
            gold_doc_idx = eval_data[q_idx]['gold_doc_idx']

            # Check retrieved chunks
            top_k_indices = res['retrieved_indices'][:k]
            relevant_found = False
            first_relevant_rank = None

            for rank, chunk_idx in enumerate(top_k_indices):
                if chunk_idx < len(chunk_to_doc) and chunk_to_doc[chunk_idx] == gold_doc_idx:
                    relevant_found = True
                    if first_relevant_rank is None:
                        first_relevant_rank = rank + 1  # 1-indexed

            recalls.append(1.0 if relevant_found else 0.0)
            reciprocal_ranks.append(1.0 / first_relevant_rank if first_relevant_rank else 0.0)

        metrics[f'Recall@{k}'] = np.mean(recalls) if recalls else 0.0

    metrics['MRR'] = np.mean(reciprocal_ranks) if reciprocal_ranks else 0.0

    return metrics


# Test metrics
metrics = compute_metrics(eval_data, chunks, c2d, results, k_values=[5, 10])
print("Baseline metrics (fixed-size, 256 chars, no overlap):")
for k, v in metrics.items():
    print(f"  {k}: {v:.3f}")

---
## Part 5: Systematic Experiments

Now run all chunking configurations and collect metrics systematically.

**TODO: This is the core of your RQ5 investigation.**

In [ ]:
# Define experimental configurations
CHUNK_METHODS = {
    'fixed': chunk_fixed_size,
    'sentence': chunk_sentence_based,
    'recursive': chunk_recursive,
}

CHUNK_SIZES = [128, 256, 512, 1024]
OVERLAPS = [0, 50]  # in characters

# TODO: Run experiments across all configurations
# Collect results in a structured format (DataFrame recommended)

results_log = []

for method_name, method_fn in CHUNK_METHODS.items():
    for size in CHUNK_SIZES:
        for overlap in OVERLAPS:
            print(f"Running: {method_name}, size={size}, overlap={overlap}...", end=" ")

            chunks, c2d, ret_results, t_idx, t_q = build_index_and_retrieve(
                documents, queries, method_fn,
                chunk_size=size, overlap=overlap,
                embed_model=embed_model, k=20  # retrieve top-20 for later reranking
            )

            metrics = compute_metrics(eval_data, chunks, c2d, ret_results,
                                       k_values=[5, 10])

            results_log.append({
                'method': method_name,
                'chunk_size': size,
                'overlap': overlap,
                'n_chunks': len(chunks),
                'recall_at_5': metrics['Recall@5'],
                'recall_at_10': metrics['Recall@10'],
                'mrr': metrics['MRR'],
                'index_time': t_idx,
                'query_time': t_q,
            })

            print(f"R@5={metrics['Recall@5']:.3f}, R@10={metrics['Recall@10']:.3f}, MRR={metrics['MRR']:.3f}")

results_df = pd.DataFrame(results_log)
print(f"\nTotal configurations tested: {len(results_df)}")
results_df

---
## Part 6: Cross-Encoder Reranking

Add a cross-encoder reranking stage and measure the improvement.

**TODO: Implement reranking for all configurations (or at least the most promising ones).**

In [ ]:
# Load cross-encoder model
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Cross-encoder loaded: ms-marco-MiniLM-L-6-v2")

In [ ]:
# TODO: Implement reranking
# For each configuration:
# 1. Retrieve top-20 with bi-encoder (already done above)
# 2. Rerank the top-20 candidates with the cross-encoder
# 3. Take top-5 after reranking
# 4. Recompute Recall@5 and MRR
# 5. Compare with and without reranking

# Example reranking for one configuration:
def rerank_results(queries, all_chunks, retrieval_results, reranker, top_k_rerank=20, final_k=5):
    """Rerank the top-k_rerank results using a cross-encoder."""
    reranked_results = []

    t0 = time.time()
    for q_idx, res in enumerate(retrieval_results):
        query = queries[q_idx]
        top_indices = res['retrieved_indices'][:top_k_rerank]

        # Create query-chunk pairs for cross-encoder
        pairs = [(query, all_chunks[idx]) for idx in top_indices if idx < len(all_chunks)]

        if not pairs:
            reranked_results.append(res)
            continue

        # Score with cross-encoder
        ce_scores = reranker.predict(pairs)

        # Sort by cross-encoder score (descending)
        scored = list(zip(top_indices[:len(pairs)], ce_scores))
        scored.sort(key=lambda x: x[1], reverse=True)

        reranked_results.append({
            'query_idx': q_idx,
            'retrieved_indices': [s[0] for s in scored[:final_k]],
            'scores': [float(s[1]) for s in scored[:final_k]],
        })

    rerank_time = time.time() - t0
    return reranked_results, rerank_time


# TODO: Run reranking experiments for your configurations
# Compare metrics before and after reranking
# Record the reranking time overhead

print("TODO: Run reranking experiments and compare with bi-encoder-only results.")
print("Hint: Use rerank_results() on the retrieval results from Part 5.")

---
## Part 7: Visualization

Create the key plots for your report.

**TODO: Create at least these visualizations:**
1. Recall@5 vs. chunk size (one curve per method)
2. Effect of overlap (grouped bar chart)
3. Before/after reranking comparison
4. Heatmap of Recall@5 across method × chunk_size

In [ ]:
import matplotlib.pyplot as plt

# TODO: Create your visualizations here
# Example structure for the main plot:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Recall@5 vs chunk size by method (no overlap)
ax = axes[0]
for method in CHUNK_METHODS:
    subset = results_df[(results_df['method'] == method) & (results_df['overlap'] == 0)]
    ax.plot(subset['chunk_size'], subset['recall_at_5'], marker='o', label=method)
ax.set_xlabel('Chunk Size (characters)')
ax.set_ylabel('Recall@5')
ax.set_title('Recall@5 vs. Chunk Size (no overlap)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: MRR vs chunk size by method (no overlap)
ax = axes[1]
for method in CHUNK_METHODS:
    subset = results_df[(results_df['method'] == method) & (results_df['overlap'] == 0)]
    ax.plot(subset['chunk_size'], subset['mrr'], marker='s', label=method)
ax.set_xlabel('Chunk Size (characters)')
ax.set_ylabel('MRR')
ax.set_title('MRR vs. Chunk Size (no overlap)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# TODO: Add more plots:
# - Effect of overlap
# - Reranking before/after comparison
# - Heatmap of results across all configurations

---
## Part 8: Failure Analysis

Examine cases where retrieval fails to find the relevant chunk.

**TODO: Identify and classify 3-5 failure examples from your best configuration.**

In [ ]:
# TODO: Failure analysis
# For your best chunking configuration:
# 1. Find queries where the correct chunk was NOT in top-5
# 2. Look at what WAS retrieved instead
# 3. Classify the failure:
#    - Wrong chunk: high similarity but irrelevant content (lexical trap)
#    - Boundary split: answer spans two chunks, neither is complete
#    - Ranked too low: correct chunk retrieved but outside top-k

# Example structure:
print("TODO: Implement failure analysis")
print("For each failure, show:")
print("  - The query")
print("  - The expected answer")
print("  - What was retrieved (top-3 chunks)")
print("  - Why it failed (your classification)")

---
## Summary

**TODO: Fill in your findings after running all experiments.**

| Finding | Your Result |
|---------|-------------|
| Best chunk size | *TODO* |
| Best chunking method | *TODO* |
| Does overlap help? | *TODO* |
| Does reranking help? | *TODO* |
| Reranking time overhead | *TODO* |
| Most common failure type | *TODO* |

**Practical recommendation:**

*TODO: Based on your results, what chunking configuration and retrieval strategy would you recommend for a production RAG pipeline?*

---

### Key Takeaways

1. Chunking is a critical decision that directly affects retrieval quality
2. The bi-encoder retrieves fast; the cross-encoder reranks accurately — the two-stage pattern is standard
3. There is a sweet spot for chunk size — too small fragments meaning, too large dilutes the embedding
4. Document structure matters — respecting boundaries (sentence, paragraph) generally helps
5. Systematic evaluation beats intuition — measure, don't guess